# Searching Rubin DP2 for supernova impostor and LBV candidates

**Author:** Mojgan Aghakhanloo (University of Virginia)  
**Runs on:** Rubin Science Platform Notebook aspect (data.lsst.cloud), recommended image, Data Preview 2  
**Package:** [`impostorsearch`](../README.md)

This notebook searches the DP2 difference-imaging catalogs for eruptive massive stars: luminous blue variables (LBVs) and supernova (SN) impostors in nearby galaxies. It follows a *transient-first* strategy:

1. **Stage 1:** query every `DiaObject` with at least `min_ndiasources` detections, strip by strip in RA.
2. **Stage 2:** associate each one with a host in the HECATE nearby-galaxy catalog ([Kovlakas et al. 2021](https://doi.org/10.1093/mnras/stab1799)), keeping objects within 2 host radii but offset from the nucleus (to avoid AGN).
3. **Stage 3:** retrieve forced photometry from `ForcedSourceOnDiaObject`, joined to `Visit` for observation times, and apply the selection criteria:
   - variation of at least 1 mag;
   - near a galaxy but offset from its nucleus;
   - peak absolute magnitude between -18 and -8, using the HECATE distance.

All thresholds are in `SearchConfig`, and each run writes them into the output metadata.

## 1. Setup
Run once in an RSP terminal: `git clone https://github.com/Aghakhanloo/lsst-impostor-search.git && pip install --user -e lsst-impostor-search/rsp_search`. Then restart the kernel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from impostorsearch import SearchConfig, load_hecate, standardize_hecate
from impostorsearch.queries import tap_client
from impostorsearch.pipeline import run_search
from impostorsearch.plots import plot_lightcurve

cfg = SearchConfig()          # edit thresholds here
service = tap_client(cfg.schema)
cfg

## 2. Load HECATE
Downloads from VizieR (J/MNRAS/506/1896) on first use. To avoid repeated downloads, save a local copy and pass its path. **Check the printed columns and the size unit the first time**: the code assumes `R1` is in arcmin (`cfg.hecate_r1_unit`).

In [ ]:
hecate_raw = load_hecate()            # or load_hecate("hecate.fits")
print(len(hecate_raw), "galaxies")
print(hecate_raw.colnames)
hosts = standardize_hecate(hecate_raw, cfg)
hosts[:5]

## 3. Test on one RA strip first
This checks query times and the number of matches before running the whole footprint.

In [ ]:
cands, metrics, phot = run_search(service, hecate_raw, cfg,
                                  ra_range=(50.0, 60.0), out_dir="outputs/test_strip")
print(f"{len(metrics)} matched objects, {len(cands)} candidates")
cands["diaObjectId", "host_name", "host_dist_mpc", "host_norm_sep",
      "amplitude_mag", "peak_abs_mag"]

## 4. Whole DP2 footprint

In [ ]:
cands, metrics, phot = run_search(service, hecate_raw, cfg,
                                  ra_range=(0.0, 360.0), out_dir="outputs/dp2_full")
print(f"{len(metrics)} matched objects, {len(cands)} candidates")

## 5. How many objects pass each criterion?

In [ ]:
for c in ["pass_offset", "pass_amplitude", "pass_absmag", "is_candidate"]:
    print(f"{c:15s} {int(np.sum(metrics[c])):7d}")

## 6. Light curves of the top candidates

In [ ]:
for row in cands[:12]:
    plot_lightcurve(phot, row["diaObjectId"], row=row)
    plt.show()

## Caveats
- Amplitudes use direct-image forced photometry, which includes host light, so they are conservative.
- Absolute magnitudes are not corrected for extinction.
- DP2 data are proprietary to Rubin data-rights holders: do not commit outputs to a public repository.